# Lab 5: Generative Models
*   **Course**: CM52065 Natural Language Processing
*   **Author**: Dr. Andrew Barnes

## Overview
This week we will be exploring three methods of text generation.

The learning objectives for this lab are as follows:

1. Implement an N-Gram generator.
2. Implement a pretrained GPT generator.
3. Compare and contrast temperature settings.

## Preparation

### 1.1 Imports
Let's start by getting all our imports in order.

In [1]:
import nltk
import string
import random
from nltk.corpus import shakespeare
from nltk.util import ngrams
from collections import defaultdict, Counter

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### 1.2 Dataset

Now we will load and preprocess a few works of Shakespeare to use in our training regime.

In [2]:
caesar = " ".join(nltk.corpus.gutenberg.words('shakespeare-caesar.txt')).lower()
hamlet = " ".join(nltk.corpus.gutenberg.words('shakespeare-hamlet.txt')).lower()
macbeth = " ".join(nltk.corpus.gutenberg.words('shakespeare-macbeth.txt')).lower()

caesar = caesar.translate(str.maketrans('', '', string.punctuation))
hamlet = hamlet.translate(str.maketrans('', '', string.punctuation))
macbeth = macbeth.translate(str.maketrans('', '', string.punctuation))

text = caesar + hamlet + macbeth

In [3]:
ctok = nltk.word_tokenize(caesar.lower())
htok = nltk.word_tokenize(hamlet.lower())
mtok = nltk.word_tokenize(macbeth.lower())

tokens = ctok + htok + mtok

In [4]:
print(tokens[:100])

['the', 'tragedie', 'of', 'julius', 'caesar', 'by', 'william', 'shakespeare', '1599', 'actus', 'primus', 'scoena', 'prima', 'enter', 'flauius', 'murellus', 'and', 'certaine', 'commoners', 'ouer', 'the', 'stage', 'flauius', 'hence', 'home', 'you', 'idle', 'creatures', 'get', 'you', 'home', 'is', 'this', 'a', 'holiday', 'what', 'know', 'you', 'not', 'being', 'mechanicall', 'you', 'ought', 'not', 'walke', 'vpon', 'a', 'labouring', 'day', 'without', 'the', 'signe', 'of', 'your', 'profession', 'speake', 'what', 'trade', 'art', 'thou', 'car', 'why', 'sir', 'a', 'carpenter', 'mur', 'where', 'is', 'thy', 'leather', 'apron', 'and', 'thy', 'rule', 'what', 'dost', 'thou', 'with', 'thy', 'best', 'apparrell', 'on', 'you', 'sir', 'what', 'trade', 'are', 'you', 'cobl', 'truely', 'sir', 'in', 'respect', 'of', 'a', 'fine', 'workman', 'i', 'am', 'but']


## Part 1: N-Gram Model

The first approach we will take to generating text is through building an N-Gram model using the corpus we loaded above.

To make our lives easier we can use the `ngram` component of the `nltk` library (imported above).

### Building the N-Gram Model

To build the n-gram model we can take a count of all possible n-grams.

> NOTE: I am using the term n-gram here because we want to keep n flexible for experimentation.

I have provided a simple function below which will take a series of tokens and build a count of all n-grams.

In [14]:
def build_ngram_model(tokens, n):
  model = defaultdict(Counter)
  for gram in ngrams(tokens, n):
    context = gram[:-1]
    next_word = gram[-1]
    model[context][next_word] += 1

  return model

Try this method out below using some trial sets of tokens to get a feel for what it is doing.

In [15]:
model = build_ngram_model(['a', 'b', 'c', 'b', 'a', 'b', 'd', 'a', 'b', 'd', 'c'], 3)

In [17]:
print("Example context:")
example_contexts = list(model.keys())
print(example_contexts)
print("For context bigram: ('a', 'b') we see the following counts " + str(model[example_contexts[0]]))

Example context:
[('a', 'b'), ('b', 'c'), ('c', 'b'), ('b', 'a'), ('b', 'd'), ('d', 'a')]
For context bigram: ('a', 'b') we see the following counts Counter({'d': 2, 'c': 1})


### N-Gram Generator

Now we need to create a generator using the N-Gram model we deveoped above.

See below a sample implementation of a text generator using the model defined above. It continuously samples words from the most recent n-gram until either a unseen sample is reached (OOV) or we reach length.

In [18]:
def generate_text(model, seed, n, length=50):
  current = tuple(seed)
  output = list(seed)
  for _ in range(length):
    if current not in model:
      break
    next_words = model[current]
    words = list(next_words.keys())
    weights = list(next_words.values())
    next_word = random.choices(words, weights=weights)[0]
    output.append(next_word)
    current = tuple(output[-(n-1):])
  return " ".join(output)

See below sample usage of the model defined above to take an input of `a` and `b`. Run this a few times to see how variety is introduced.

### Shakespearian N-Grams

Great, now we have the core functionality it's time to train a Shakespearian N-Gram model to generate text.

In the space below write a script which will do the following tasks:
1. Generate 3 N-Gram models where n = 2, 3 and 4 using the Shakespeare corpus.
2. Generate text using each of these models by passing them the firs `n-1` tokens each.

In [32]:
for n in [2, 3, 4]:
  print(f"== TRAINING {n}-GRAM")
  model = build_ngram_model(tokens, n)
  print(f"== GENERATING USING {n}-GRAM")
  print(generate_text(model, tokens[:n-1], n, length=25))

== TRAINING 2-GRAM
== GENERATING USING 2-GRAM
the melting spirits our king one incapable of normandy i stand in her owne liues meanes speake to see the end but at all welcome la
== TRAINING 3-GRAM
== GENERATING USING 3-GRAM
the tragedie of macbeth marry he closes with you good cornelius and you know that i see a robustious pery wig pated fellow teare a passion to
== TRAINING 4-GRAM
== GENERATING USING 4-GRAM
the tragedie of julius caesar by william shakespeare 1603 actus primus scoena prima enter flauius murellus and certaine commoners ouer the stage flauius hence home you idle creatures


> **Question:** By replaying the generation multiple times, what do you notice about the stability of the outputs?

## Part 2: Transformer Model

In this next section we will introduce ourselves to a new library `transformers` and use this library to load a pre-trained generative model (GPT2) and tokeniser. We will then use this model to generate text.

### Loading the model

To begin, we will load both the model and tokeniser using the `transformers` library.

In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Generate Text

Now we can use this model to generate text, to do this we use the `model.generate(...)` function which requires us to define a temperature and several other crucial parameters.

Below is an example call to the generate function.

In [40]:
prompt = "Why do cows moo?"
#prompt = "Explain machine learning"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids

gen_tokens = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.5,
    max_length=50,
)
gen_text = tokenizer.batch_decode(gen_tokens)[0]
print(gen_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Explain machine learning and machine learning algorithms to understand the real world and how to improve it.

The following is a list of the topics covered in this course:

Machine Learning

Machine Learning is a powerful and very well-known


Using the space below modify the prompt, temperature and max_length of the generator and answer the following questions:

> **Question:** What effect does a small temperature have on the output?
> **Question:** What effect does a high temperature have on the output?
> **Question:** Does max_length effect the style of the output?

## Optional Exercise (Challenging)

Using the HuggingFace platform and the `transformers` library select and compare three different generative pretrained transformers on the same prompts and identify their strengths and weaknesses of each.

See [HuggingFace](https://huggingface.co/docs/transformers/).

## Wrap-up
Well done on finishing up this week's lab! I hope you were able to practice what we learnt in the lectures and now have a practical understanding of how to use generative models.